In [1]:
import os
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop","QPR","数据")

os.chdir(desktop_path)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import pennylane as qml
from pennylane import numpy as pnp # PennyLane 的 numpy
import numpy as np # 原生 numpy 用于绘图
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# 1. 加载数据与划分训练集
# ==========================================
# 加载原始数据
inputs_raw = np.loadtxt('9input.txt', dtype=complex)
outputs_raw = np.loadtxt('9output.txt')
# 转换为 Torch 张量
inputs_all = torch.tensor(inputs_raw, dtype=torch.complex128)
outputs_all = torch.tensor(outputs_raw, dtype=torch.long)

num_samples = inputs_all.shape[0]
num_qubits = int(np.log2(inputs_all.shape[1]))
print(f"量子比特数: {num_qubits} | 总样本数: {num_samples}")


量子比特数: 9 | 总样本数: 2500


In [3]:



# ==========================================
# 2. 定义 QCNN 电路 (PennyLane)
# ==========================================
def qcnn_ansatz(num_qubits, params):
    """Ansatz of the QCNN model
    Repetitions of the convolutional and pooling blocks
    until only 2 wires are left unmeasured
    """

    # Convolution block
    def conv(wires, params, index):
        if len(wires) % 2 == 0:
            groups = wires.reshape(-1, 2)
        else:
            groups = wires[:-1].reshape(-1, 2)
            qml.RY(params[index], wires=int(wires[-1]))
            index += 1

        for group in groups:
            qml.CNOT(wires=[int(group[0]), int(group[1])])
            for wire in group:
                qml.RY(params[index], wires=int(wire))
                index += 1
        if len(wires) > 2:
                # 提取中间错开的部分
                interleaved_wires = wires[1:] 
                if len(interleaved_wires) % 2 == 0:
                    inter_groups = interleaved_wires.reshape(-1, 2)
                else:
                    inter_groups = interleaved_wires[:-1].reshape(-1, 2)
                    # 对最后一个未配对的比特应用单比特旋转（可选，增加参数灵活性）
                    qml.RY(params[index], wires=int(interleaved_wires[-1]))
                    index += 1

                for group in inter_groups:
                    qml.CNOT(wires=[int(group[0]), int(group[1])])
                    for wire in group:
                        qml.RY(params[index], wires=int(wire))
                        index += 1
        return index

    # Pooiling block
    def pool(wires, params, index):
        # Process wires in pairs: measure one and conditionally rotate the other.
        for wire_pool, wire in zip(wires[0::2], wires[1::2]):
            m_0 = qml.measure(int(wire_pool))
            qml.cond(m_0 == 0, qml.RX)(params[index],     wires=int(wire))
            qml.cond(m_0 == 1, qml.RX)(params[index + 1], wires=int(wire))
            index += 2
            # Remove the measured wire from active wires.
            wires = np.delete(wires, np.where(wires == wire_pool))

        # If an odd wire remains, apply a RX rotation.
        if len(wires) % 2 != 0:
            qml.RX(params[index], wires=int(wires[-1]))
            index += 1

        return index, wires

    # Initialize active wires and parameter index.
    active_wires = np.arange(num_qubits)
    index = 0

    # Initial layer: apply RY to all wires.
    for wire in active_wires:
        qml.RY(params[index], wires=int(wire))
        index += 1

    # Repeatedly apply convolution and pooling until there are 2 unmeasured wires
    while len(active_wires) > 2:
        # Convolution
        index = conv(active_wires, params, index)
        # Pooling
        index, active_wires = pool(active_wires, params, index)
        qml.Barrier()

    # Final layer: apply RY to the remaining active wires.
    for wire in active_wires:
        qml.RY(params[index], wires=int(wire))
        index += 1

    return index, active_wires

num_params, output_wires = qcnn_ansatz(num_qubits, [0]*100)

@qml.qnode(qml.device("default.qubit", wires=num_qubits),interface="torch")
def qcnn_circuit(params, state):
    """QNode with QCNN ansatz and probabilities of unmeasured qubits as output"""
    # Input ground state from diagonalization
    qml.StatePrep(state, wires=range(num_qubits), normalize = True)
    # QCNN
    _, output_wires = qcnn_ansatz(num_qubits, params)

    return qml.probs([int(k) for k in output_wires])
# 自动计算参数量
def get_num_params():
    dummy_params = torch.zeros(2000)
    idx, _ = qcnn_ansatz(num_qubits, dummy_params)
    return idx

num_params = get_num_params()

# ==========================================
# 3. 损失函数与训练流程
# ==========================================
def cross_entropy_with_sharpening(pred, target, T):
    epsilon = 1e-9
    pred = torch.clamp(pred, epsilon, 1.0)
    # T 为温度，T<1 使分布更尖锐
    logits = torch.log(pred) / T
    log_probs = torch.log_softmax(logits, dim=1)
    return -torch.mean(torch.sum(target * log_probs, dim=1))

def train_qcnn(num_epochs=50, lr=0.02, T=0.5,train_size=20):
    params = torch.randn(num_params, dtype=torch.float64, requires_grad=True)
    optimizer = Adam([params], lr=lr)
    
    # 训练数据准备
    # 随机挑选 20 个训练点
    # torch.manual_seed(42) # 固定随机种子以便复现
    train_indices = torch.randperm(num_samples)[:train_size]
    X_train = inputs_all[train_indices]
    Y_train = outputs_all[train_indices]
    # Y_train_onehot = F.one_hot(Y_train, num_classes=4).to(torch.float64)
    Y_onehot = F.one_hot(Y_train, num_classes=4).to(torch.float64)

    loss_curve = []
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        
        # 为了防止内存溢出，按批次或循环处理
        batch_preds = []
        for x in X_train:
            batch_preds.append(qcnn_circuit(params, x))
        
        preds = torch.stack(batch_preds)
        loss = cross_entropy_with_sharpening(preds, Y_onehot, T)
        
        loss.backward()
        optimizer.step()
        
        loss_curve.append(loss.item())
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f}")
            
    return params.detach(), loss_curve

# ==========================================





In [4]:
# 4. 执行与可视化
# ==========================================

# 设定实验参数
import numpy as np
import torch
import matplotlib.pyplot as plt

# 1. 定义要测试的训练集大小列表
train_sizes = [5, 10, 20, 50, 100]  # 你可以根据数据集总量调整这些数值

num_runs = 10  # 每个尺寸下运行的次数（建议至少5次以观察标准差）
num_epochs = 150
# 存储最终统计结果
final_results = []
stats_summary = {
    "sizes": train_sizes,
    "means": [],
    "stds": [],
    "raw_data": {}
}
print(f"开始不同训练集规模的对比实验...")

for size in train_sizes:
    size_accuracies = []
    print(f"\n{'='*20} 正在测试 train_size = {size} {'='*20}")
    
    for run in range(num_runs):
        # 训练模型 (传入当前的 size)
        # 注意：请确保你的 train_qcnn 函数内部会根据输入的 size 重新对数据进行采样
        trained_params, _ = train_qcnn(num_epochs=num_epochs, lr=0.05, T=1, train_size=size)
        
        # 评估模型 (在全集 inputs_all 上测试)
        with torch.no_grad():
            all_preds_list = []
            for x in inputs_all:
                pred_prob = qcnn_circuit(trained_params, x)
                all_preds_list.append(pred_prob)
            
            all_preds = torch.stack(all_preds_list)
            predicted_labels = torch.argmax(all_preds, dim=1)
            
            correct = (predicted_labels == outputs_all).sum().item()
            accuracy = (correct / len(outputs_all)) * 100
            size_accuracies.append(accuracy)
            
        print(f"  Run {run + 1}: Accuracy = {accuracy:.2f}%")
    
    # 计算当前 size 的统计信息
    avg_acc = np.mean(size_accuracies)
    std_acc = np.std(size_accuracies,ddof=1)
    final_results.append({'size': size, 'avg': avg_acc, 'std': std_acc})
    stats_summary["means"].append(avg_acc)
    stats_summary["stds"].append(std_acc)
    stats_summary["raw_data"][f"size_{size}"] = size_accuracies

# ==========================================
# 2. 绘制实验对比图
# ==========================================
sizes = [res['size'] for res in final_results]
averages = [res['avg'] for res in final_results]
stds = [res['std'] for res in final_results]

plt.figure(figsize=(10, 6))
plt.errorbar(sizes, averages, yerr=stds, fmt='-o', capsize=5, color='b', ecolor='r', label='Accuracy with Std Dev')
plt.xlabel('Training Set Size', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Impact of Training Size on Model Performance', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

开始不同训练集规模的对比实验...

==================== 正在测试 train_size = 5 ====================
Epoch 05 | Loss: 0.9372
Epoch 10 | Loss: 0.8079
Epoch 15 | Loss: 0.7103
Epoch 20 | Loss: 0.5906
Epoch 25 | Loss: 0.4810
Epoch 30 | Loss: 0.4347
Epoch 35 | Loss: 0.4169
Epoch 40 | Loss: 0.4105
Epoch 45 | Loss: 0.4062
Epoch 50 | Loss: 0.4022
Epoch 55 | Loss: 0.3955
Epoch 60 | Loss: 0.3896
Epoch 65 | Loss: 0.3833
Epoch 70 | Loss: 0.3747
Epoch 75 | Loss: 0.3676
Epoch 80 | Loss: 0.3625
Epoch 85 | Loss: 0.3597
Epoch 90 | Loss: 0.3576
Epoch 95 | Loss: 0.3544
Epoch 100 | Loss: 0.3494
Epoch 105 | Loss: 0.3404
Epoch 110 | Loss: 0.3270
Epoch 115 | Loss: 0.3131
Epoch 120 | Loss: 0.3022
Epoch 125 | Loss: 0.2963
Epoch 130 | Loss: 0.2935
Epoch 135 | Loss: 0.2908
Epoch 140 | Loss: 0.2885
Epoch 145 | Loss: 0.2874
Epoch 150 | Loss: 0.2871


KeyboardInterrupt: 

In [ ]:
# import json
# import numpy as np

# # ==========================================
# # 2. 存储数据 (优化后的版本)
# # ==========================================

# # 构建完整的存储字典，包含您要求的 raw_data
# export_data = {
#     "metadata": {
#         "num_runs": num_runs,
#         "num_epochs": num_epochs,
#         "train_sizes": train_sizes,
#     },
#     "statistics": {
#         "sizes": stats_summary["sizes"],
#         "means": stats_summary["means"],
#         "stds": stats_summary["stds"]
#     },
#     # 包含每一轮的具体准确率
#     "raw_data": stats_summary["raw_data"], 

# }

# # 写入文件
# file_name = "qcnn_experiment_results9.json"
# with open(file_name, "w", encoding="utf-8") as f:
#     # 使用 indent=4 让 JSON 文件人类可读，ensure_ascii=False 支持中文（如果有）
#     json.dump(export_data, f, indent=4, ensure_ascii=False)

# print(f"\n✅ 实验数据（含原始数据）已成功保存至: {file_name}")


✅ 实验数据（含原始数据）已成功保存至: qcnn_experiment_results9.json
